# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
DATA_PATH = Path("data/hmeq.csv")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

In [4]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns="BAD")
y = df["BAD"]

In [5]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name}: {len(part)} rows, default rate {part.mean():.3f}")

train: 3576 rows, default rate 0.199
val: 1192 rows, default rate 0.200
test: 1192 rows, default rate 0.200


In [6]:
RAW_NUM = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ",
           "CLAGE", "NINQ", "CLNO", "DEBTINC"]

def add_features(df):
    out = df.copy()
    out["DEBTINC_na"] = out["DEBTINC"].isna().astype(int)
    out["VALUE_na"] = out["VALUE"].isna().astype(int)
    out["n_missing"] = df[RAW_NUM + ["REASON", "JOB"]].isna().sum(axis=1)
    out["small_loan"] = (out["LOAN"] <= 10_000).astype(int)
    out["LTV"] = out["LOAN"] / out["VALUE"]
    out["MORTDUE_VALUE"] = out["MORTDUE"] / out["VALUE"]
    out["DEBTINC"] = out["DEBTINC"].clip(upper=60)
    for c in ["LOAN", "MORTDUE", "VALUE", "CLAGE"]:
        out[c] = np.log1p(out[c])
    out[["REASON", "JOB"]] = out[["REASON", "JOB"]].fillna("Missing")
    return out

In [7]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUM = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ",
       "CLNO", "DEBTINC", "LTV", "MORTDUE_VALUE", "n_missing"]
FLAGS = ["DEBTINC_na", "VALUE_na", "small_loan"]
CAT = ["REASON", "JOB"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
    ("flags", "passthrough", FLAGS),
])

X_train_p = pre.fit_transform(add_features(X_train)).astype("float32")
X_val_p = pre.transform(add_features(X_val)).astype("float32")
X_test_p = pre.transform(add_features(X_test)).astype("float32")
feature_names = list(pre.get_feature_names_out())

print(X_train_p.shape, X_val_p.shape, X_test_p.shape)
print("NaNs:", np.isnan(X_train_p).sum(), np.isnan(X_val_p).sum(), np.isnan(X_test_p).sum())
joblib.dump(pre, ARTIFACTS / "preprocessor.joblib")

(3576, 26) (1192, 26) (1192, 26)
NaNs: 0 0 0


['artifacts/preprocessor.joblib']

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss

torch.manual_seed(SEED)
FN_COST, FP_COST = 10_000, 1_500          # EUR (hypothetical, see REPORT.md)

y_tr, y_va, y_te = y_train.to_numpy(), y_val.to_numpy(), y_test.to_numpy()

def expected_cost(y, p, t):
    """Average EUR lost per applicant if we reject everyone with p >= t."""
    pred = p >= t
    fn = ((~pred) & (y == 1)).sum()        # approved a defaulter
    fp = (pred & (y == 0)).sum()           # rejected a good borrower
    return (FN_COST * fn + FP_COST * fp) / len(y)

THRESHOLDS = np.linspace(0.01, 0.99, 99)
def best_threshold(y, p):
    costs = [expected_cost(y, p, t) for t in THRESHOLDS]
    return THRESHOLDS[int(np.argmin(costs))]

# Naive baselines on test
base_approve_all = expected_cost(y_te, np.zeros(len(y_te)), 0.5)   # majority class
base_reject_all = expected_cost(y_te, np.ones(len(y_te)), 0.5)
print(f"baseline approve all: {base_approve_all:.0f} EUR/applicant")
print(f"baseline reject all:  {base_reject_all:.0f} EUR/applicant")

baseline approve all: 1997 EUR/applicant
baseline reject all:  1201 EUR/applicant


In [9]:
# The only hyperparameter we touch: regularisation strength C (chosen on validation)
best = None
for C in [0.01, 0.1, 1, 10]:
    m = LogisticRegression(C=C, max_iter=20000, tol=1e-10).fit(X_train_p, y_tr)
    ll = log_loss(y_va, m.predict_proba(X_val_p)[:, 1])
    if best is None or ll < best[0]:
        best = (ll, C, m)
_, C_BEST, sk_model = best

N, D = X_train_p.shape
LAM = 1.0 / (C_BEST * N)    # same L2 strength expressed for the PyTorch versions
print("best C:", C_BEST, "-> LAM:", LAM)

best C: 1 -> LAM: 0.0002796420581655481


In [10]:
Xt = torch.tensor(X_train_p)
yt = torch.tensor(y_tr, dtype=torch.float32)

w = torch.zeros(D, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
LR_MANUAL, EPOCHS_MANUAL = 1.0, 20000

for epoch in range(EPOCHS_MANUAL):
    z = Xt @ w + b                                              # 1. forward
    loss = F.binary_cross_entropy_with_logits(z, yt) \
           + 0.5 * LAM * (w ** 2).sum()                         # 2. loss (+ L2 on weights)
    loss.backward()                                             # 3. backward (autograd)
    with torch.no_grad():                                       # 4. manual step
        w -= LR_MANUAL * w.grad
        b -= LR_MANUAL * b.grad
    w.grad.zero_()
    b.grad.zero_()
print("manual final loss:", float(loss))

manual final loss: 0.28400909900665283


/tmp/ipykernel_2092/1885179622.py:18: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("manual final loss:", float(loss))


In [12]:
class LogReg(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.linear = nn.Linear(d, 1)
    def forward(self, x):
        return self.linear(x).squeeze(1)       # logits

model = LogReg(D)
nn.init.zeros_(model.linear.weight)
nn.init.zeros_(model.linear.bias)

optimizer = torch.optim.Adam([
    {"params": [model.linear.weight], "weight_decay": LAM},   # L2 on weights
    {"params": [model.linear.bias], "weight_decay": 0.0},     # intercept not penalised
], lr=0.05)
criterion = nn.BCEWithLogitsLoss()

for epoch in range(5000):
    optimizer.zero_grad()
    loss = criterion(model(Xt), yt)
    loss.backward()
    optimizer.step()
print("nn final loss:", float(loss))

nn final loss: 0.2807687819004059


In [13]:
def probs_manual(X):
    with torch.no_grad():
        return torch.sigmoid(torch.tensor(X) @ w + b).numpy()

def probs_nn(X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(X))).numpy()

predictors = {
    "sklearn": lambda X: sk_model.predict_proba(X)[:, 1],
    "manual_torch": probs_manual,
    "nn_torch": probs_nn,
}

rows, thresholds = [], {}
for name, predict in predictors.items():
    p_val, p_test = predictor_val, predictor_test = predict(X_val_p), predict(X_test_p)
    t = best_threshold(y_va, p_val)                 # threshold chosen on VALIDATION
    thresholds[name] = float(t)
    rows.append({"model": name, "AUC": roc_auc_score(y_te, p_test),
                 "log_loss": log_loss(y_te, p_test), "threshold": t,
                 "cost_per_applicant": expected_cost(y_te, p_test, t)})

prior = np.full(len(y_te), y_tr.mean())
rows.append({"model": "baseline: approve all", "AUC": 0.5, "log_loss": log_loss(y_te, prior),
             "threshold": np.nan, "cost_per_applicant": base_approve_all})
rows.append({"model": "baseline: reject all", "AUC": 0.5, "log_loss": log_loss(y_te, prior),
             "threshold": np.nan, "cost_per_applicant": base_reject_all})
results = pd.DataFrame(rows).round(4)
print(results.to_string(index=False))

# Do the three agree?
p_sk, p_m, p_n = (f(X_test_p) for f in predictors.values())
print("max |sklearn - manual|:", np.abs(p_sk - p_m).max())
print("max |sklearn - nn|    :", np.abs(p_sk - p_n).max())
print("max |manual - nn|     :", np.abs(p_m - p_n).max())

# Save everything the app needs (no retraining there)
joblib.dump(sk_model, ARTIFACTS / "sklearn_model.joblib")
torch.save({"w": w.detach(), "b": b.detach()}, ARTIFACTS / "manual_torch.pt")
torch.save(model.state_dict(), ARTIFACTS / "nn_torch.pt")
np.savez(ARTIFACTS / "splits.npz", X_val=X_val_p, y_val=y_va, X_test=X_test_p, y_test=y_te)
joblib.dump({"feature_names": feature_names, "thresholds": thresholds,
             "fn_cost": FN_COST, "fp_cost": FP_COST, "C": C_BEST}, ARTIFACTS / "meta.joblib")
results.to_csv(ARTIFACTS / "results.csv", index=False)

                model    AUC  log_loss  threshold  cost_per_applicant
              sklearn 0.8829    0.3075       0.15            551.1745
         manual_torch 0.8829    0.3075       0.15            551.1745
             nn_torch 0.8829    0.3075       0.15            551.1745
baseline: approve all 0.5000    0.4999        NaN           1996.6443
 baseline: reject all 0.5000    0.4999        NaN           1200.5034
max |sklearn - manual|: 0.0004490614
max |sklearn - nn|    : 0.0004786253
max |manual - nn|     : 0.00016158819
